## 01. 데이터 수집


### 분석 대상

- **지역:** 서울특별시 성북구, 동대문구
- **기간:** 2023년 1월 ~ 2026년 7월
- **주택 유형:** 연립·다세대주택
- **분석 목적:** 개별 임대차 계약이 주변 유사 주택 시세 대비 적정한 수준인지 판단하기 위한 기초 데이터 구축

### 주요 작업

1. 라이브러리 및 환경 설정
2. 데이터 수집 범위 및 기간 설정
3. 국토교통부 실거래 데이터 수집
4. 데이터 수집 결과 검증



# 1. 라이브러리 및 환경 설정

공공데이터 API 인증키는 코드에 직접 입력하지 않고,  
Google Colab의 `Secrets` 기능을 통해 불러옴.

In [ ]:
import time
import requests
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

from google.colab import userdata

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

SERVICE_KEY = userdata.get("DATA_GO_KR_KEY")

if SERVICE_KEY is None:
    raise ValueError(
        "Colab Secrets에 'DATA_GO_KR_KEY'가 등록되어 있지 않습니다."
    )

print("라이브러리 및 API 인증키 불러오기 완료")

라이브러리 및 API 인증키 불러오기 완료


# 2. 데이터 수집 범위 및 기간 설정

*   분석 대상 지역: 성북구, 동대문구

*   분석 기간: 2023년 1월부터 2026년 7월까지 월별 데이터를 조회합니다.




In [ ]:
URL = (
    "http://apis.data.go.kr/1613000/"
    "RTMSDataSvcRHRent/getRTMSDataSvcRHRent"
)

LAWD_CODES = {
    "11290": "성북구",
    "11230": "동대문구"
}

START_YM = "202301"
END_YM = "202607"


def create_month_range(start_ym: str, end_ym: str) -> list[str]:
    """YYYYMM 형식의 시작월과 종료월 사이의 월 목록을 생성합니다."""
    months = []

    start_year = int(start_ym[:4])
    end_year = int(end_ym[:4])

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            year_month = f"{year}{month:02d}"

            if start_ym <= year_month <= end_ym:
                months.append(year_month)

    return months


deal_ymds = create_month_range(START_YM, END_YM)

print(f"조회 기간: {deal_ymds[0]} ~ {deal_ymds[-1]}")
print(f"조회 개월 수: {len(deal_ymds)}개월")
print(f"대상 자치구 수: {len(LAWD_CODES)}개")
print(f"예상 API 요청 수: {len(deal_ymds) * len(LAWD_CODES)}회")

조회 기간: 202301 ~ 202607
조회 개월 수: 43개월
대상 자치구 수: 2개
예상 API 요청 수: 86회


# 3. 국토교통부 실거래 데이터 수집


<각 자치구와 계약연월별로 API를 호출하여 임대차 계약 정보를 수집>

함께 기록되는 목록:

- 데이터가 조회된 자치구
- API 요청 기준 연월
- API 오류 또는 경고 내역
- 한 요청에서 1,000건을 초과하여 추가 페이지 조회가 필요한 경우

In [ ]:
def fetch_rent_data(
    service_key: str,
    lawd_codes: dict,
    deal_ymds: list[str],
    num_rows: int = 1000,
    sleep_seconds: float = 0.2
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """연립·다세대 전월세 실거래가 데이터를 월별로 수집합니다."""

    rows = []
    errors = []

    total_requests = len(lawd_codes) * len(deal_ymds)
    request_count = 0

    for lawd_code, district_name in lawd_codes.items():
        for deal_ymd in deal_ymds:
            request_count += 1

            params = {
                "serviceKey": service_key,
                "LAWD_CD": lawd_code,
                "DEAL_YMD": deal_ymd,
                "numOfRows": num_rows,
                "pageNo": 1
            }

            try:
                response = requests.get(
                    URL,
                    params=params,
                    timeout=20
                )
                response.raise_for_status()

                root = ET.fromstring(response.content)

                result_code = root.findtext(".//resultCode")
                result_message = root.findtext(".//resultMsg")

                if result_code != "000":
                    errors.append({
                        "법정동코드": lawd_code,
                        "자치구": district_name,
                        "조회연월": deal_ymd,
                        "오류내용": result_message
                    })
                    continue

                total_count = int(root.findtext(".//totalCount") or 0)

                for item in root.iter("item"):
                    row = {
                        child.tag: child.text
                        for child in item
                    }

                    row["_구"] = district_name
                    row["_조회연월"] = deal_ymd
                    rows.append(row)

                if total_count > num_rows:
                    errors.append({
                        "법정동코드": lawd_code,
                        "자치구": district_name,
                        "조회연월": deal_ymd,
                        "오류내용": (
                            f"전체 {total_count}건으로 "
                            f"{num_rows}건 초과: 추가 페이징 필요"
                        )
                    })

            except Exception as error:
                errors.append({
                    "법정동코드": lawd_code,
                    "자치구": district_name,
                    "조회연월": deal_ymd,
                    "오류내용": str(error)
                })

            if request_count % 10 == 0:
                print(
                    f"{request_count}/{total_requests}개 요청 완료"
                )

            time.sleep(sleep_seconds)

    data = pd.DataFrame(rows)
    error_log = pd.DataFrame(errors)

    return data, error_log


raw_df, error_log = fetch_rent_data(
    service_key=SERVICE_KEY,
    lawd_codes=LAWD_CODES,
    deal_ymds=deal_ymds
)

print()
print(f"수집된 전체 행 수: {len(raw_df):,}건")
print(f"오류 및 경고 수: {len(error_log):,}건")

10/86개 요청 완료
20/86개 요청 완료
30/86개 요청 완료
40/86개 요청 완료
50/86개 요청 완료
60/86개 요청 완료
70/86개 요청 완료
80/86개 요청 완료

수집된 전체 행 수: 28,207건
오류 및 경고 수: 0건


# 4. 데이터 수집 결과 검증

In [ ]:
print("데이터 크기:", raw_df.shape)

print("\n조회연월 범위")
print(
    raw_df["_조회연월"].min(),
    "~",
    raw_df["_조회연월"].max()
)

print("\n자치구별 수집 건수")
display(
    raw_df["_구"]
    .value_counts()
    .rename_axis("자치구")
    .reset_index(name="거래건수")
)

print("\n월별 수집 건수")
monthly_raw_count = (
    raw_df.groupby(["_조회연월", "_구"])
    .size()
    .unstack(fill_value=0)
)

display(monthly_raw_count.tail(12))

데이터 크기: (28207, 20)

조회연월 범위
202301 ~ 202607

자치구별 수집 건수


,자치구,거래건수
0,성북구,16677
1,동대문구,11530



월별 수집 건수


_구,동대문구,성북구
_조회연월,,
202508,271,329
202509,222,333
202510,204,308
202511,269,391
202512,389,535
202601,406,520
202602,296,427
202603,250,431
202604,212,349


In [ ]:
if error_log.empty:
    print("API 오류 또는 경고가 없습니다.")
else:
    print("API 오류 및 경고 내역")
    display(error_log)

API 오류 또는 경고가 없습니다.


##  데이터 저장

In [ ]:
raw_df.to_csv('dataset_ML1조.csv', index=False, encoding='utf-8-sig')